# CNN — Detecção de Células Cervicais (SipakMed) v2

**Dataset:** SipakMed — 5 classes de células do colo do útero  
**Modelo:** MobileNetV2 com Transfer Learning  
**Output:** `cnn_cervical.h5` para uso no Streamlit

### Melhorias v2
- **CLAHE** — equalização adaptativa de histograma por canal (melhora contraste celular)
- **Normalização de coloração** — corrige variação de corante entre imagens
- **EfficientNetB0** como alternativa ao MobileNetV2 (melhor accuracy/custo)

### Pré-requisito
Faça upload do dataset SipakMed no Google Drive antes de rodar.
Estrutura esperada:
```
MyDrive/sipakmed/
  im_Dyskeratotic/im_Dyskeratotic/*.bmp
  im_Koilocytotic/im_Koilocytotic/*.bmp
  im_Metaplastic/im_Metaplastic/*.bmp
  im_Parabasal/im_Parabasal/*.bmp
  im_Superficial-Intermediate/im_Superficial-Intermediate/*.bmp
```
### Ativar GPU: **Runtime → Change runtime type → T4 GPU → Save**

## 1. Monta o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

SIPAKMED_PATH = '/content/drive/MyDrive/sipakmed'
OUTPUT_PATH   = '/content/drive/MyDrive/cervical_artifacts'
os.makedirs(OUTPUT_PATH, exist_ok=True)

assert os.path.exists(SIPAKMED_PATH), f'SipakMed não encontrado em: {SIPAKMED_PATH}'

classes = sorted([d for d in os.listdir(SIPAKMED_PATH)
                  if os.path.isdir(os.path.join(SIPAKMED_PATH, d))])
print(f'Classes: {classes}')

## 2. Instala dependências e imports

In [ ]:
!pip install -q opencv-python-headless

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 3. Função de pré-processamento com CLAHE

**CLAHE** (Contrast Limited Adaptive Histogram Equalization):
- Equaliza o histograma em regiões locais da imagem (não globalmente)
- Realça bordas e estruturas celulares sem saturar as cores
- Reduz a variação de contraste entre imagens de diferentes equipamentos/corantes
- Aplicado em cada canal RGB separadamente

In [ ]:
def apply_clahe(img_uint8: np.ndarray) -> np.ndarray:
    """
    Aplica CLAHE em cada canal RGB da imagem.

    Args:
        img_uint8: Array uint8 shape (H, W, 3) com valores 0-255

    Returns:
        Array uint8 com CLAHE aplicado
    """
    # Cria o objeto CLAHE
    # clipLimit=2.0 — limita amplificação de ruído
    # tileGridSize=(8,8) — divide a imagem em blocos 8x8 para equalizar localmente
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    # Aplica CLAHE em cada canal separadamente (R, G, B)
    channels = cv2.split(img_uint8)                    # Separa em 3 canais
    eq_channels = [clahe.apply(ch) for ch in channels] # Equaliza cada canal
    return cv2.merge(eq_channels)                       # Junta de volta


def preprocess_image(fpath: str, img_size=(224, 224)) -> np.ndarray:
    """
    Carrega, pré-processa e normaliza uma imagem BMP.

    Pipeline:
    1. Carrega com OpenCV (suporte nativo a BMP)
    2. BGR → RGB
    3. Redimensiona para 224x224
    4. Aplica CLAHE para realçar estruturas celulares
    5. Normaliza para [0, 1]

    Returns:
        Array float32 shape (224, 224, 3) ou None se falhar
    """
    img = cv2.imread(fpath)
    if img is None:
        return None

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   # BGR → RGB
    img = cv2.resize(img, img_size)               # Redimensiona
    img = apply_clahe(img)                        # CLAHE — realça contraste
    img = img.astype(np.float32) / 255.0          # Normaliza para [0, 1]

    return img


# Visualiza o efeito do CLAHE em uma imagem de exemplo
sample_class = 'im_Dyskeratotic'
sample_path  = os.path.join(SIPAKMED_PATH, sample_class, sample_class)
sample_file  = os.path.join(sample_path, sorted(os.listdir(sample_path))[0])

img_orig = cv2.cvtColor(cv2.imread(sample_file), cv2.COLOR_BGR2RGB)
img_orig = cv2.resize(img_orig, (224, 224))
img_clahe_uint8 = apply_clahe(img_orig)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_orig)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(img_clahe_uint8)
axes[1].set_title('Com CLAHE')
axes[1].axis('off')
plt.suptitle('Efeito do CLAHE na imagem celular', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Carrega e prepara as imagens

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

BINARY_MAP = {
    'im_Dyskeratotic':             1,
    'im_Koilocytotic':             1,
    'im_Metaplastic':              0,
    'im_Parabasal':                0,
    'im_Superficial-Intermediate': 0,
}

images = []
labels = []

for class_folder, label in BINARY_MAP.items():
    class_path = os.path.join(SIPAKMED_PATH, class_folder, class_folder)

    if not os.path.isdir(class_path):
        print(f'  Pasta não encontrada: {class_path}')
        continue

    bmp_files = [f for f in os.listdir(class_path) if f.endswith('.bmp')]
    print(f'  {class_folder}: {len(bmp_files)} imagens')

    for fname in bmp_files:
        fpath = os.path.join(class_path, fname)
        img = preprocess_image(fpath, IMG_SIZE)   # Carrega com CLAHE
        if img is not None:
            images.append(img)
            labels.append(label)

X = np.array(images, dtype=np.float32)
y = np.array(labels, dtype=np.int32)

print(f'\nTotal: {len(X)} imagens')
print(f'Normais (0): {(y==0).sum()} | Anômalas (1): {(y==1).sum()}')

## 5. Split e Data Augmentation

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Treino: {len(X_train)} | Validação: {len(X_val)} | Teste: {len(X_test)}')

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=30,        # Mais rotação — células não têm orientação fixa
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],  # Variação de brilho — simula diferentes equipamentos
    fill_mode='nearest'
)

train_gen = datagen.flow(X_train, y_train, batch_size=BATCH_SIZE)
print(f'Steps por epoch: {len(X_train) // BATCH_SIZE}')

## 6. Constrói o modelo

In [ ]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)   # Mais neurônios que v1
x = layers.Dropout(0.4)(x)
x = layers.Dense(64, activation='relu')(x)    # Camada extra
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc')
    ]
)

model.summary()

## 7. Treinamento — Fase 1

In [ ]:
n_normal = (y_train == 0).sum()
n_anomal = (y_train == 1).sum()
total    = len(y_train)
class_weight = {
    0: total / (2 * n_normal),
    1: total / (2 * n_anomal)
}
print(f'Class weights: {class_weight}')

callbacks = [
    EarlyStopping(monitor='val_recall', patience=7, restore_best_weights=True, mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_PATH, 'cnn_cervical_best.h5'),
        monitor='val_recall',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

print('=== FASE 1: Treinamento das camadas novas ===')
history1 = model.fit(
    train_gen,
    epochs=25,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1
)

## 8. Fine-tuning — Fase 2

In [ ]:
base_model.trainable = True

# Descongela as últimas 50 camadas (mais que v1)
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-6),  # LR ainda menor
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.Recall(name='recall'), keras.metrics.AUC(name='auc')]
)

print('=== FASE 2: Fine-tuning das últimas 50 camadas ===')
history2 = model.fit(
    train_gen,
    epochs=25,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1
)

## 9. Avaliação

In [ ]:
print('=== AVALIAÇÃO FINAL ===')
test_loss, test_acc, test_recall, test_auc = model.evaluate(X_test, y_test, verbose=0)
print(f'Accuracy: {test_acc:.4f}')
print(f'Recall:   {test_recall:.4f}  ← mais importante!')
print(f'AUC-ROC:  {test_auc:.4f}')

# Threshold 0.35 — favorece recall (menos falsos negativos)
y_pred_proba = model.predict(X_test).ravel()
y_pred = (y_pred_proba >= 0.35).astype(int)

print('\nRelatório (threshold=0.35):')
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Anômala (1)']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anômala'],
            yticklabels=['Normal', 'Anômala'])
plt.title('Matriz de Confusão — CNN Cervical v2')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

acc  = history1.history['accuracy']     + history2.history['accuracy']
vacc = history1.history['val_accuracy'] + history2.history['val_accuracy']
rec  = history1.history['recall']       + history2.history['recall']
vrec = history1.history['val_recall']   + history2.history['val_recall']

axes[0].plot(acc,  label='Treino')
axes[0].plot(vacc, label='Validação')
axes[0].set_title('Accuracy')
axes[0].legend()

axes[1].plot(rec,  label='Treino')
axes[1].plot(vrec, label='Validação')
axes[1].set_title('Recall')
axes[1].legend()

plt.suptitle('Curvas de Treinamento v2 (com CLAHE)', fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Salva o modelo

In [ ]:
model_path = os.path.join(OUTPUT_PATH, 'cnn_cervical.h5')
model.save(model_path)

size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f'Modelo salvo: {model_path} ({size_mb:.1f} MB)')
print('\n✅ Pronto! Coloque em: app/models/artifacts/cnn_cervical.h5')

In [ ]:
# Download direto sem passar pelo Drive
from google.colab import files
files.download(model_path)